# NPath Text Mining: N-Gram Analysis for Classification

This notebook performs n-gram analysis and discriminative feature extraction for text classification using the opinions dataset.

**Target**:  column (Claim, Evidence, Counterclaim, etc.)  
**Features**:  column (student opinions/arguments)

## Analysis Pipeline:
1. Load and clean opinions dataset
2. Generate n-grams (bigrams, trigrams, 4-grams)
3. Calculate discriminative scores for classification
4. Export results for model training

---
**Dataset**: ~34K student opinion texts  
**GitHub**: https://github.com/hincaltopcuoglu/Npath-text-mining

In [ ]:
# @title 🔄 FORCE SYNC: Pull Latest Changes from GitHub
# Run this FIRST to get the latest code updates
import os
from pathlib import Path

print("🔄 Force Sync: Pulling latest changes from GitHub...")

# GitHub repository details
GITHUB_USERNAME = "hincaltopcuoglu"
REPO_NAME = "Npath-text-mining"
BRANCH = "master"

repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if Path(REPO_NAME).exists():
    print(f"📂 Repository exists, pulling latest changes...")
    os.chdir(REPO_NAME)
    
    # Force pull latest changes
    !git fetch origin
    !git reset --hard origin/{BRANCH}
    
    print(f"✅ Successfully synced to latest {BRANCH} branch")
    
    # Show recent commits
    print("📝 Recent changes:")
    !git log --oneline -5
    
    os.chdir("..")
else:
    print(f"📥 Repository not found, cloning...")
    !git clone -b {BRANCH} {repo_url}
    

# Verify files
required_files = ["colab_ngram_analysis.py", "data/raw/opinions.csv", "pattern_ranking.py"]
print("
🔍 File verification:")
for file_path in required_files:
    if Path(file_path).exists():
        print(f"  ✅ {file_path}")
    else:
        print(f"  ❌ Missing: {file_path}")

print("
🎯 Ready to run analysis with latest code!")

In [ ]:
# @title 💾 Setup Persistence: Save to Google Drive
# Mount Google Drive for persistent storage across sessions

from google.colab import drive
import os
from pathlib import Path

print("💾 Setting up Google Drive persistence...")

# Mount Google Drive
try:
    drive.mount("/content/drive", force_remount=True)
    print("✅ Google Drive mounted successfully!")
except Exception as e:
    print(f"❌ Drive mount failed: {e}")
    print("💡 Make sure to click the link above and authorize access.")

# Create persistent directory
PERSISTENT_DIR = "/content/drive/MyDrive/NPath_Analysis"
RESULTS_BACKUP = f"{PERSISTENT_DIR}/results_backup"

os.makedirs(PERSISTENT_DIR, exist_ok=True)
os.makedirs(RESULTS_BACKUP, exist_ok=True)

print(f"📁 Persistent storage: {PERSISTENT_DIR}")
print(f"💾 Results backup: {RESULTS_BACKUP}")

print("
🎯 Your analysis results will be saved to Google Drive!")
print("💡 You can resume work from any Colab session.")

In [ ]:
# @title 📊 Resume Previous Analysis
# Load saved results from Google Drive

from pathlib import Path
import pandas as pd
import shutil

PERSISTENT_DIR = "/content/drive/MyDrive/NPath_Analysis"
RESULTS_BACKUP = f"{PERSISTENT_DIR}/results_backup"

print("📊 Checking for saved analysis results...")

# Check if Drive is mounted
if not Path("/content/drive").exists():
    print("❌ Google Drive not mounted. Run the persistence setup cell first.")
else:
    # Restore previous results
    if Path(RESULTS_BACKUP).exists() and list(Path(RESULTS_BACKUP).iterdir()):
        print("📋 Restoring previous results from Drive...")
        
        # Remove current results if they exist
        if Path("colab_results").exists():
            shutil.rmtree("colab_results")
        
        # Copy from Drive
        shutil.copytree(RESULTS_BACKUP, "colab_results")
        
        # Show what was restored
        restored_files = list(Path("colab_results").glob("*.csv"))
        print(f"✅ Restored {len(restored_files)} result files:")
        for file_path in sorted(restored_files):
            print(f"  📄 {file_path.name}")
        
        print("🎯 Ready to view results or continue analysis!")
    else:
        print("❌ No saved results found in Drive.")
        print("💡 Run an analysis first, then results will be automatically saved.")

In [ ]:
# @title 🔄 Quick Sync: Check for Updates
# Optional: Use if you want to check for updates without force pull
import os
from pathlib import Path

# GitHub repository details
GITHUB_USERNAME = "hincaltopcuoglu"  # @param {type:"string"}
REPO_NAME = "Npath-text-mining"     # @param {type:"string"}
BRANCH = "master"                   # @param {type:"string"}

repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if Path(REPO_NAME).exists():
    print(f"🔄 Checking for updates on {BRANCH} branch...")
    os.chdir(REPO_NAME)
    
    # Check status
    !git fetch origin
    status = !git status -uno
    
    if "Your branch is behind" in str(status):
        print("📥 Updates available! Run the FORCE SYNC cell to update.")
        !git log --oneline HEAD..origin/{BRANCH}
    else:
        print("✅ Already up to date!")
    
    os.chdir("..")
else:
    print(f"❌ Repository not found. Run FORCE SYNC first.")

print("
💡 Tip: Use FORCE SYNC cell to always get latest changes!")

In [ ]:
# @title 📥 Download Missing Files (Alternative)
# If GitHub sync fails, manually download required files
import os
from pathlib import Path

# Create directories
os.makedirs("data/raw", exist_ok=True)

# Check what files we need
missing_files = []
required_files = [
    ("colab_ngram_analysis.py", "https://raw.githubusercontent.com/hincaltopcuoglu/Npath-text-mining/master/colab_ngram_analysis.py"),
    ("data/raw/opinions.csv", "https://raw.githubusercontent.com/hincaltopcuoglu/Npath-text-mining/master/data/raw/opinions.csv")
]

for filename, url in required_files:
    if not Path(filename).exists():
        missing_files.append((filename, url))

if missing_files:
    print(f"📥 Downloading {len(missing_files)} missing files...")
    import urllib.request
    
    for filename, url in missing_files:
        print(f"  Downloading {filename}...")
        try:
            urllib.request.urlretrieve(url, filename)
            print(f"  ✅ {filename}")
        except Exception as e:
            print(f"  ❌ Failed to download {filename}: {e}")
else:
    print("✅ All required files are available!")

# Final verification
print("
🔍 Final file check:")
for filename, _ in required_files:
    status = "✅" if Path(filename).exists() else "❌"
    print(f"  {status} {filename}")

In [ ]:
# @title 📦 Install Dependencies
# Install required packages for Colab
print("📦 Installing dependencies...")

# Core ML/data science packages
!pip install -q pandas numpy scikit-learn matplotlib seaborn plotly

# NLP packages
!pip install -q nltk tqdm

# Download NLTK data (comprehensive download)
import nltk
print("📥 Downloading NLTK data...")
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("omw-1.4", quiet=True)

print("✅ Dependencies and NLTK data installed!")

# Test imports
try:
    import pandas as pd
    import numpy as np
    from sklearn.feature_extraction.text import CountVectorizer
    from nltk.util import ngrams
    from nltk.tokenize import word_tokenize
    # Test NLTK functionality
    test_text = "This is a test sentence."
    tokens = word_tokenize(test_text)
    print(f"✅ All imports successful! Test tokenization: {tokens}")
except Exception as e:
    print(f"❌ Import error: {e}")
    print("💡 Try restarting the runtime if issues persist")

In [ ]:
# @title 🚀 Run N-Gram Analysis
# Execute the complete n-gram analysis pipeline

# Import the analyzer
from colab_ngram_analysis import ColabNgramAnalyzer

# Configuration parameters
N_VALUES = [2, 3, 4]        # @param {type:"raw"}
MIN_FREQ = 5                # @param {type:"integer"} # Minimum frequency for n-grams
MIN_SUPPORT = 10            # @param {type:"integer"} # Minimum support for discriminative analysis
TOP_K = 500                 # @param {type:"integer"} # Top k discriminative n-grams per class
BATCH_SIZE = 1000           # @param {type:"integer"} # Processing batch size

print("🚀 Starting N-Gram Analysis Pipeline")
print("=" * 50)
print(f"N-grams: {N_VALUES}")
print(f"Min frequency: {MIN_FREQ}")
print(f"Min support: {MIN_SUPPORT}")
print(f"Top k per class: {TOP_K}")
print(f"Batch size: {BATCH_SIZE}")
print("=" * 50)

# Initialize analyzer
analyzer = ColabNgramAnalyzer(
    data_path="data/raw/opinions.csv",
    text_col="text",
    target_col="type"
)

# Run complete analysis
analyzer.run_complete_analysis(
    n_values=N_VALUES,
    min_freq=MIN_FREQ,
    min_support=MIN_SUPPORT,
    top_k=TOP_K,
    batch_size=BATCH_SIZE
)

print("
✅ Analysis Complete!")

In [ ]:
# @title 🎯 Advanced Pattern Ranking & Analysis
# Rank patterns using multiple scoring metrics (nPath-style)

# Import the pattern ranker
from pattern_ranking import PatternRanker

# Configuration parameters
N_VALUES_RANKING = [2, 3, 4]    # @param {type:"raw"}
TOP_K_RANKING = 20               # @param {type:"integer"} # Top k patterns per class
MIN_CONFIDENCE = 0.1              # @param {type:"number"} # Minimum confidence threshold
LIFT_THRESHOLD = 1.5              # @param {type:"number"} # Minimum lift threshold

print("🎯 Starting Advanced Pattern Ranking Analysis")
print("=" * 60)
print(f"N-grams: {N_VALUES_RANKING}")
print(f"Top k per class: {TOP_K_RANKING}")
print(f"Min confidence: {MIN_CONFIDENCE}")
print(f"Lift threshold: {LIFT_THRESHOLD}")
print("=" * 60)

# Initialize ranker
ranker = PatternRanker(results_dir="colab_results")

# Update thresholds
ranker.min_confidence = MIN_CONFIDENCE
ranker.lift_threshold = LIFT_THRESHOLD

# Run complete ranking analysis
ranker.run_complete_ranking_analysis(
    n_values=N_VALUES_RANKING,
    top_k=TOP_K_RANKING
)

print("
✅ Pattern Ranking Complete!")
print("📊 Rankings saved to colab_results/*gram_rankings.csv")
print("🖼️  Visualizations saved as top_*gram_patterns_comparison.png")

In [ ]:
# @title 💾 Auto-Save Results to Drive
# Automatically backup results after analysis

from pathlib import Path
import shutil
import time

PERSISTENT_DIR = "/content/drive/MyDrive/NPath_Analysis"
RESULTS_BACKUP = f"{PERSISTENT_DIR}/results_backup"

print("💾 Auto-saving results to Google Drive...")

# Check if Drive is available
if not Path("/content/drive").exists():
    print("⚠️  Google Drive not mounted. Results not saved to Drive.")
    print("💡 Run the persistence setup cell to enable Drive saving.")
else:
    # Check if results exist
    if Path("colab_results").exists():
        try:
            # Remove old backup
            if Path(RESULTS_BACKUP).exists():
                shutil.rmtree(RESULTS_BACKUP)
            
            # Copy current results
            shutil.copytree("colab_results", RESULTS_BACKUP)
            
            # Count saved files
            saved_files = list(Path(RESULTS_BACKUP).glob("*"))
            csv_files = list(Path(RESULTS_BACKUP).glob("*.csv"))
            png_files = list(Path(RESULTS_BACKUP).glob("*.png"))
            
            timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
            print(f"✅ Results auto-saved to Drive at {timestamp}")
            print(f"📊 {len(csv_files)} CSV files, {len(png_files)} images saved")
            print(f"📁 Location: {RESULTS_BACKUP}")
            
            print("🎯 Your work is safe! You can resume from any Colab session.")
        except Exception as e:
            print(f"❌ Auto-save failed: {e}")
    else:
        print("❌ No results to save. Run analysis first.")

print("
💡 Tip: Results are automatically saved after each analysis run!")

# 📋 Ready to Run!

## 🚀 Launch Instructions:
1. Open [Google Colab](https://colab.research.google.com/)
2. **File → Open notebook → GitHub**
3. Enter: 
4. Select: 
5. **Run all cells** sequentially

## 📊 Expected Output:
- Discriminative n-grams for text classification
- CSV files ready for ML model training
- Automatic sync back to GitHub

---
**Your NPath analysis is ready! 🎯**